# 06.01 · Retrieval concepts and RAG architectures

**Module 06 · Documents, Embeddings & RAG** · Part 2 (Retrieval)

| | |
|---|---|
| ⏱️ Estimated time | 60–90 min |
| 📋 Prerequisites | Module 02 (tools and agents), Lesson 03.06 (embeddings, briefly). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** why a model needs retrieval for private or recent facts, and what "grounded" means.
2. **Name** the two phases of RAG: **indexing** (load → split → embed → store) and **retrieval + generation**.
3. **Build** a whole (tiny) RAG pipeline offline in a dozen lines with `mastery.rag`.
4. **Compare** 2-step, agentic and hybrid RAG on latency, cost, control and flexibility.
5. **Find** the current LangChain RAG documentation (the old `langchain/rag` URL now redirects).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval) | Building a knowledge base; 2-step vs agentic vs hybrid RAG (the table this lesson builds on) |
| [Semantic search (knowledge base)](https://docs.langchain.com/oss/python/langchain/knowledge-base) | Load → split → embed → store, with pypdf and `RecursiveCharacterTextSplitter` |
| [Component architecture](https://docs.langchain.com/oss/python/langchain/component-architecture) | Where loaders, splitters, embeddings, vector stores and retrievers fit |
| [Agentic RAG (LangGraph)](https://docs.langchain.com/oss/python/langgraph/agentic-rag) | The custom-graph version of agentic RAG (Module 07 builds graphs) |
| [RAG with Deep Agents](https://docs.langchain.com/oss/python/deepagents/rag) | Where `/oss/python/langchain/rag` redirects now; Deep Agents come in Module 10 |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langchain-text-splitters>=1.1.3,<2" "pypdf>=6.20.1,<7"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-text-splitters", "pypdf",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/06-rag/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out. Set these BEFORE importing chromadb/fastembed.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma: no usage telemetry.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")   # Windows: HF cache can't symlink without Developer Mode.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")        # HF tokenizers' fork warning in Jupyter.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# tqdm (used by the model downloader) asks for ipywidgets to draw a progress bar; text output is fine.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
# Step 5: The course corpus: 24 Markdown files + 8 PDFs (23 pages) about fictional Nimbus weather stations.
import json

RAG_DIR = REPO_ROOT / "data" / "samples" / "rag"
CORPUS_DIR = RAG_DIR / "corpus"
GOLDEN = [json.loads(line) for line in (RAG_DIR / "golden-set.jsonl").read_text(encoding="utf-8").splitlines()]
print("corpus files:", sum(1 for p in CORPUS_DIR.rglob("*") if p.is_file()), "| golden questions:", len(GOLDEN))

**You should now see** `corpus files: 32 | golden questions: 20`.

The corpus is **fictional** on purpose: "Orbita Labs" and its Nimbus weather stations don't exist, so no model can know the answers from training. If a model gets them right, it *must* have used retrieval. (Generated by `_maintainer/tools/m06/data.py`; you'll meet every file in 06.02.)

---
## Part 1 · Why retrieval?

A model only knows its training data. Ask it about your company's return policy and it will either say it doesn't know (good) or invent something plausible (bad). **Retrieval-augmented generation (RAG)** fixes that: find the relevant passages first, put them in the prompt, and ask the model to answer **only** from them.

> 💸 Uses a few thousand input tokens of your free-tier quota (the retrieved chunks are input). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: ask the model WITHOUT retrieval. It has never seen the Nimbus Pro.
QUESTION = "What does error E12 mean on the Nimbus Pro weather station?"
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    print(llm.invoke(QUESTION + " Answer in one sentence.").text)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) either "I don't have information about..." or a confident guess (a sensor fault, a Wi-Fi error...). Both are wrong for us: the right answer is in a PDF manual the model never saw.

---
## Part 2 · Phase 1: indexing (load → split → embed → store)

| Step | What | LangChain building block | This module |
|---|---|---|---|
| **Load** | Files → `Document(page_content, metadata)` | document loaders | 06.02 |
| **Split** | Long documents → chunks that fit and match well | text splitters | 06.03 |
| **Embed** | Chunk text → vector of numbers | `Embeddings` | 06.04 |
| **Store** | Vectors + text + metadata, searchable by similarity | `VectorStore` | 06.05 |

Indexing runs **once** (and again when documents change), not on every question. Here is the whole phase, offline, with `mastery.rag` and the offline `HashingEmbeddings` from Module 03 (word-based, no download; real embeddings in 06.04):

In [ ]:
# 🧪 Offline: the whole indexing phase.
from mastery.memory import HashingEmbeddings
from mastery.rag import build_index, chunk_stats, load_corpus, retrieve, split_documents

docs = load_corpus(CORPUS_DIR)                                   # Load: 24 Markdown files + 23 PDF pages
chunks = split_documents(docs, "markdown", chunk_size=800, chunk_overlap=100)   # Split
store = build_index(chunks, HashingEmbeddings(dims=1024))        # Embed + store (in memory)

print("documents:", len(docs), "| chunks:", chunk_stats(chunks))
for doc in retrieve(store, QUESTION, k=3):                       # Phase 2 starts here: retrieve
    print(f"{doc.metadata['score']:.3f}  {doc.metadata['source']}  page={doc.metadata.get('page')}")

**You should now see** `documents: 47` (24 Markdown + 23 PDF pages), the chunk statistics, and three hits, among them `manuals/nimbus-pro-manual.pdf page=4` (the troubleshooting page). The product page `products/nimbus-pro.md` may rank first: it shares the words *Nimbus Pro weather station* with the question. The score is cosine similarity: higher is closer; with word-based embeddings it counts shared words, not meaning (06.04).

---
## Part 3 · Phase 2: 2-step RAG (retrieve, then generate)

**2-step RAG** always does the same two things: (1) retrieve the top-k chunks for the question, (2) one model call with those chunks in the prompt. Predictable: **one** retrieval, **one** model call.

In [ ]:
# 🧪 Offline: 2-step RAG with a scripted "reader" (it answers by quoting the best retrieved sentence).
from langchain_core.prompts import ChatPromptTemplate

from mastery.rag import CitedAnswer, extractive_reader, format_citations, format_context
from mastery.testing import ScriptedChatModel

PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from the sources. Cite each fact with a word-for-word quote. "
               "If the sources don't contain the answer, say you don't know."),
    ("human", "Sources:\n{context}\n\nQuestion: {question}"),
])
reader = ScriptedChatModel(responses=[extractive_reader()], repeat_last=True)
rag_chain = PROMPT | reader.with_structured_output(CitedAnswer)

def two_step(question: str, chain=rag_chain, k: int = 4) -> CitedAnswer:
    hits = retrieve(store, question, k=k)                                    # Step 1: retrieve
    return chain.invoke({"context": format_context(hits), "question": question})   # Step 2: generate

print(format_citations(two_step(QUESTION)))

**You should now see** `Error E12 means the rain gauge is blocked: clean the funnel.` and `[1] nimbus-pro-manual.pdf, p. 4: "..."`. The scripted reader can't write; it only picks a sentence. That makes it honest (the quote is always real) but dull. A real model writes better answers from the same prompt:

In [ ]:
# 💸 Live: the same 2-step chain with a real model.
if HAVE_GROQ_KEY:
    live_chain = PROMPT | llm.with_structured_output(CitedAnswer)
    print(format_citations(two_step(QUESTION, chain=live_chain)))
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) a one-sentence answer about the blocked rain gauge and one or two citations from `nimbus-pro-manual.pdf`. 06.06 adds quote verification and the "don't know" policy.

---
## Part 4 · Agentic RAG (the model decides when and what to search)

In **agentic RAG**, retrieval is a **tool**. The agent decides whether to search, with which query, and whether to search again. Flexible (follow-up searches, several sources), but less predictable: 1–N tool calls and 2–N model calls.

In [ ]:
# 🧪 Offline: an agent with a search tool, driven by a script (search once, then answer).
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage

from mastery.agents import format_transcript
from mastery.rag import make_retriever_tool

search_docs = make_retriever_tool(store, name="search_docs", k=3,
                                  description="Search the Orbita Labs product docs. Input: a short query.")
script = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "search_docs", "args": {"query": "Nimbus Pro error E12"}, "id": "s1"}]),
    AIMessage("E12 means the rain gauge is blocked; clean the funnel (nimbus-pro-manual.pdf, page 4)."),
])
rag_agent = create_agent(script, tools=[search_docs])
out = rag_agent.invoke({"messages": [HumanMessage(QUESTION)]})
print(format_transcript(out["messages"], max_chars=110))

**You should now see** four messages: your question, the model's `search_docs` tool call, the tool result (numbered sources), and the final answer.

### What just happened?

Same index, same question, two architectures. In 2-step **your code** called `retrieve()`; in agentic RAG **the model** called the tool. The tool returns the chunks twice: as numbered text for the model (`content`), and as the original `Document` list (`artifact`) so your code can check citations later (06.07).

---
## Part 5 · Choosing an architecture

| | **2-step RAG** | **Agentic RAG** | **Hybrid RAG** |
|---|---|---|---|
| Flow | Always retrieve → generate | Model decides to search (0–N times) | 2-step plus checks: rewrite the query, grade the context, verify the answer, retry |
| Model calls | 1 | 2–N | 2–4 |
| Latency / cost | Lowest, predictable | Variable | Medium |
| Control | High | Lower | High |
| Good for | FAQ bots, docs Q&A where every question needs the docs | Mixed chat + lookups, several sources, follow-up questions | Ambiguous queries, quality-critical answers |
| In this module | 06.06 | 06.07 | 06.08 (rewrite, re-rank), 06.09 (grade) |

A practical rule: **start with 2-step**. Move to agentic when questions often need no retrieval, several sources, or several searches. Add hybrid steps when you can **measure** that they help (06.09).

In [ ]:
# 🧪 Offline: count model calls per architecture for the same question.
print("2-step  model calls:", len(reader.calls), "(one structured call per question)")
print("agentic model calls:", len(script.calls), "| tool calls:",
      sum(len(m.tool_calls) for m in out["messages"] if isinstance(m, AIMessage)))

**You should now see** `2-step model calls: 1` and `agentic model calls: 2 | tool calls: 1`. With a real model the agent may also search twice. Every extra call is latency and tokens.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Tutorial code imports `langchain.chains.RetrievalQA` / `create_retrieval_chain` | LangChain 0.x chains, moved to `langchain-classic` | Build the chain yourself (prompt + model + your `retrieve()`), as in Part 3 |
| `/oss/python/langchain/rag` shows a Deep Agents page | The URL now redirects (2026) | Start from the [Retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval) and [knowledge base](https://docs.langchain.com/oss/python/langchain/knowledge-base) pages |
| The model answers without the sources | The prompt doesn't forbid it | "Answer ONLY from the sources" + a "don't know" rule + verified citations (06.06) |
| `FileNotFoundError: data/samples/rag/...` | Old checkout | `git pull` and re-run the Before-you-start block |

## ⚠️ Production notes

> ⚠️ **Production note: RAG moves the risk, it doesn't remove it.**
> The model can still misread a chunk, or the retriever can miss the right one. Measure both (06.09): retrieval (did we find it?) and generation (did we use it faithfully?).

> ⚠️ **Production note: documents are untrusted input.**
> Anyone who can edit a document can put instructions into your prompt (indirect prompt injection). Treat retrieved text as data and verify outputs (06.06).

## 🏋️ Exercises

1. **(Easy)** Run `two_step()` with "How long are raw readings kept?". Which file is cited?
2. **(Medium)** Ask `two_step("Who founded Orbita Labs?")`. What does the reader answer, and why is that the right behaviour?
3. **(Medium)** Change `k=4` to `k=1` in `two_step` and try question 1 again. What could go wrong with `k=1` on a question whose answer is in two files?
4. **(Stretch, 💸)** Give the agent a real model: `create_agent(llm, tools=[search_docs])` and ask a question that needs no search ("Say hi"). Does it call the tool?

<details>
<summary><b>Show solutions</b></summary>

**1.** `policies/data-retention.md` (or `support/faq-data-export.md`: both say 24 months).

**2.** `I don't know: the answer is not in the provided sources.` The founder isn't in the corpus; "founded" appears in no retrieved chunk, so the reader abstains instead of guessing.

**3.** With `k=1` you get one chunk: if the answer needs two files (or the best chunk is ranked second), the model can't see it. Typical `k` is 3–6.

**4.** Usually not: agentic RAG skips retrieval when it isn't needed. That's its advantage over 2-step (and its risk: it may also skip it when it *is* needed).
</details>

## ✅ Checkpoint

- [ ] I can name the four indexing steps and the two phases of RAG.
- [ ] I built and queried an index offline.
- [ ] I can explain when to choose 2-step, agentic or hybrid RAG.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Where does `https://docs.langchain.com/oss/python/langchain/rag` lead today? As of 2026-10-12 it redirects (308) to the Deep Agents RAG page; the current entry points are the [Retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval) page, the [knowledge base](https://docs.langchain.com/oss/python/langchain/knowledge-base) tutorial, [agentic RAG with LangGraph](https://docs.langchain.com/oss/python/langgraph/agentic-rag) and [RAG with Deep Agents](https://docs.langchain.com/oss/python/deepagents/rag). Check them (`curl -sI <url>` in Git Bash shows redirects) and log what you find in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- RAG = retrieve relevant chunks, then answer **only** from them, with citations.
- Indexing (load → split → embed → store) runs once; retrieval + generation runs per question.
- 2-step: predictable and cheap. Agentic: flexible, more calls. Hybrid: 2-step + checks you can measure.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 06.02 · Loading and parsing documents (PDF, HTML, Markdown)** → [`06-02-document-loaders.ipynb`](06-02-document-loaders.ipynb).

[Module 06 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)